# EEG Pipeline

Full pipeline from raw file to cleaned epochs:
load → inspect → filter → re-reference → ICA → ICLabel → artifact removal → epochs

**Available channels:** `PO7 O1 Oz O2 PO8 PO4 POz PO3 P5 P3 P1 Pz P2 P4 P6 CP6 CP2 CPz CP1 CP5 C5 FC5 FC1 Cz FC2 F4 FC6 C6 Fz F3 AF3 AF4`

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

PROJECT_ROOT = Path("../.." ).resolve()
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from eeg.io import load_bci2k
from eeg.inspect import print_summary
from eeg.preprocessing import (
    bandpass_filter,
    label_components,
    make_epochs,
    remove_artifacts,
    rereference_average,
    run_ica,
)
from eeg.viz import (
    plot_before_after,
    plot_channels,
    plot_channels_psd,
    plot_ica_components,
    plot_psd,
    plot_raw_traces,
    plot_stim_channel,
)

DATA_FILE = PROJECT_ROOT / "data/eeg/raw/MET000bGridFixedS001R02.dat"

# --- parameters (edit these) ---
CHANNELS        = ["Oz", "Fz", "Pz"]   # channels for granular plots and before/after
TRACE_DURATION  = 200.0                  # seconds shown in time-domain plots
L_FREQ, H_FREQ  = 1.0, 55.0            # bandpass cutoffs (Hz)
FILTER_ORDER    = 4
N_ICA           = 24                   # ICA components
EYE_THRESHOLD   = 0.70                 # ICLabel eye-blink exclusion threshold
MUSCLE_THRESHOLD = 0.50                # ICLabel muscle exclusion threshold
EPOCH_TMIN      = 0.0                  # epoch start relative to stim onset (s)
EPOCH_TMAX      = 3.0                  # epoch end (s)

---
## 1. Load and inspect

In [ ]:
raw = load_bci2k(DATA_FILE)
print_summary(raw)

### Raw traces (first 20 channels)

In [ ]:
plot_raw_traces(raw, duration=TRACE_DURATION)

### Stimulus channel

In [ ]:
plot_stim_channel(raw,TRACE_DURATION)

### Selected channels — time domain

In [ ]:
plot_channels(raw, CHANNELS, duration=TRACE_DURATION)

### Selected channels — PSD

In [ ]:
plot_channels_psd(raw, CHANNELS)

### Full PSD (all channels)

In [ ]:
plot_psd(raw)

---
## 2. Bandpass filter

In [ ]:
bandpass_filter(raw, l_freq=L_FREQ, h_freq=H_FREQ, order=FILTER_ORDER)
print(f"Filtered: {L_FREQ}–{H_FREQ} Hz, order {FILTER_ORDER}")

### PSD after filtering

In [ ]:
plot_psd(raw)
plot_channels_psd(raw, CHANNELS,55)

---
## 3. Re-reference to average

In [ ]:
rereference_average(raw)

---
## 4. ICA

Extended Infomax, `n_components=24`. Takes ~60–90 s.

In [ ]:
ica = run_ica(raw, n_components=N_ICA)
print(ica)

---
## 5. ICLabel

In [ ]:
labels = label_components(raw, ica)

for i, (label, proba) in enumerate(zip(labels["labels"], labels["y_pred_proba"])):
    print(f"IC {i:02d}: {label} ({proba.max():.0%})")

### Component properties

Edit `COMPONENT_INDICES` to inspect any components.

In [ ]:
COMPONENT_INDICES = [0, 1]

plot_ica_components(raw, ica, COMPONENT_INDICES, labels=labels)

---
## 6. Remove artifact components

In [ ]:
raw_clean, excluded = remove_artifacts(
    raw, ica, labels,
    eye_threshold=EYE_THRESHOLD,
    muscle_threshold=MUSCLE_THRESHOLD,
)
print(f"Excluded ICs: {excluded}")
for i in excluded:
    print(f"  IC {i:02d}: {labels['labels'][i]} ({labels['y_pred_proba'][i].max():.0%})")

### Before / after ICA cleaning

In [ ]:
plot_before_after(raw, raw_clean, CHANNELS, duration=TRACE_DURATION)

---
## 7. Create epochs

Locked to STI 014 rising edges, `EPOCH_TMIN` to `EPOCH_TMAX` seconds. Expect 104 trials.

In [ ]:
epochs = make_epochs(raw_clean, tmin=EPOCH_TMIN, tmax=EPOCH_TMAX)
print(epochs)
print(f"Shape: {epochs.get_data().shape}  (trials x channels x samples)")